In [ ]:
!pip install -q 'transformers>=4.40.0'
!pip install -q datasets evaluate accelerate albumentations roboflow

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
  print(result.stdout)
  print('GPU is available! You are good to go.')

In [ ]:
import os, cv2, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as npatches
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import(
    AutoImageProcessor,
    Mask2FormerForUniversalSegmentation
)
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
  print(f'   GPU: {torch.cuda.get_device_name(0)}')
  print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from roboflow import Roboflow

USE_DEMO_DATA = True
ROBOFLOW_API_KEY   = 'TdOKixM39Uh1tATTadka'
ROBOFLOW_WORKSPACE = 'mds-workspace-pgksy'
ROBOFLOW_PROJECT   = 'coco_facades-fngvm'
ROBOFLOW_VERSION   = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download("png-mask-semantic")
DATA_BASE      = dataset.location

In [ ]:
TRAIN_IMG_DIR  = os.path.join(DATA_BASE, 'train')
TRAIN_MASK_DIR = os.path.join(DATA_BASE, 'train')
VALID_IMG_DIR  = os.path.join(DATA_BASE, 'valid')
VALID_MASK_DIR = os.path.join(DATA_BASE, 'valid')

In [ ]:
def create_demo_dataset(base_dir, num_train=60, num_valid=15, img_size=256):
  np.random.seed(42)
  for split, count in [('train', num_train), ('valid', num_valid)]:
    img_dir = os.path.join(base_dir, split, 'images')
    mask_dir = os.path.join(base_dir, split, 'masks')
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)
    for i in range(count):
      bg    = np.random.randint(50, 200, 3)
      image = np.ones((img_size, img_size, 3), dtype=np.uint8) * bg
      mask  = np.zeros((img_size, img_size), dtype=np.uint8)
      for _ in range(np.random.randint(1, 4)):
         cx  = np.random.randint(30, img_size - 30)
         cy  = np.random.randint(30, img_size - 30)
         r   = np.random.randint(15, 50)
         col = tuple(int(c) for c in np.random.randint(0, 255, 3))
         cv2.circle(image, (cx, cy), r, col, -1)
         cv2.circle(mask,  (cx, cy), r, 1,   -1)
      fname = f'img_{i:04d}.png'
      cv2.imwrite(os.path.join(img_dir,  fname), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
      cv2.imwrite(os.path.join(mask_dir, fname), mask)

  print(f' Demo dataset: {num_train} train + {num_valid} valid images saved to {base_dir}')

In [ ]:
class_list = project.classes
ID2LABEL = {i: name for i, name in enumerate(class_list)}
LABEL2ID = {name: i for i, name in enumerate(class_list)}
NUM_CLASSES = len(ID2LABEL)

print(f'Detected {NUM_CLASSES} classes.')
print(f'Classes: {ID2LABEL}')

In [ ]:
import matplotlib.cm as cm

def preview_samples(img_dir, mask_dir, n=4):
    all_files = sorted(os.listdir(img_dir))
    image_files = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg')) and '_mask' not in f][:n]

    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1: axes = [axes]

    cmap = cm.get_cmap('tab20', NUM_CLASSES)
    preview_colors = (cmap(np.arange(NUM_CLASSES))[:, :3] * 255).astype(np.uint8)

    for i, fname in enumerate(image_files):
        img = cv2.cvtColor(cv2.imread(os.path.join(img_dir, fname)), cv2.COLOR_BGR2RGB)

        mask_fname = os.path.splitext(fname)[0] + '_mask.png'
        mask_path = os.path.join(mask_dir, mask_fname)

        if not os.path.exists(mask_path):
            mask_path = os.path.join(mask_dir, fname)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f' Mask not found for {fname}'); continue

        overlay = img.copy()
        unique_classes = np.unique(mask)
        for cls_id in unique_classes:
            if cls_id == 0: continue
            color_idx = int(cls_id) % NUM_CLASSES
            mask_indices = (mask == cls_id)
            overlay[mask_indices] = (overlay[mask_indices] * 0.4 + preview_colors[color_idx] * 0.6).astype(np.uint8)

        axes[i][0].imshow(img);               axes[i][0].set_title(f'Image: {fname[:15]}...');   axes[i][0].axis('off')
        axes[i][1].imshow(mask, cmap='nipy_spectral'); axes[i][1].set_title('Semantic Mask');    axes[i][1].axis('off')
        axes[i][2].imshow(overlay);           axes[i][2].set_title('Multi-class Overlay'); axes[i][2].axis('off')

    plt.suptitle('Dataset Preview (Multi-class Support)', fontsize=14, y=1.01)
    plt.tight_layout(); plt.show()

    if len(image_files) > 0:
        test_mask = cv2.imread(os.path.join(mask_dir, os.path.splitext(image_files[0])[0] + '_mask.png'), 0)
        if test_mask is not None:
            print(f'Mask unique values: {np.unique(test_mask).tolist()}')

In [ ]:
preview_samples(TRAIN_IMG_DIR, TRAIN_MASK_DIR, n=4)

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, img_dir, mask_dir, processor):
      self.img_dir  = img_dir
      self.mask_dir = mask_dir
      self.processor = processor
      self.images = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
      return len(self.images)

    def __getitem__(self, idx):
      fname = self.images[idx]
      image = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, fname)), cv2.COLOR_BGR2RGB)
      mask_path = os.path.join(self.mask_dir, fname)
      if not os.path.exists(mask_path):
        mask_path = os.path.join(self.mask_dir, os.path.splitext(fname)[0] + '.png')
      mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
      mask = np.clip(mask, 0, NUM_CLASSES - 1)

      unique_ids = np.unique(mask)
      instance_id_to_semantic_id = {int(uid): int(uid) for uid in unique_ids}

      encoded = self.processor(
          images=image,
          segmentation_maps=mask,
          instance_id_to_semantic_id=instance_id_to_semantic_id,
          return_tensors='pt'
      )

      return {
            'pixel_values': encoded['pixel_values'].squeeze(0),
            'mask_labels' : encoded['mask_labels'][0],
            'class_labels': encoded['class_labels'][0],
        }

In [ ]:
processor = AutoImageProcessor.from_pretrained(
    'facebook/mask2former-swin-small-ade-semantic',
    ignore_index = 255,
    reduce_labels = False
)

In [ ]:
train_dataset = SegmentationDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, processor)
valid_dataset = SegmentationDataset(VALID_IMG_DIR, VALID_MASK_DIR, processor)

In [ ]:
sample = train_dataset[0]
print(f'\npixel_values shape: {sample["pixel_values"].shape}  (channels, height, width)')
print(f'mask_labels shape : {sample["mask_labels"].shape}   (num_masks, height, width)')
print(f'class_labels      : {sample["class_labels"].tolist()} (class ID per mask)')

In [ ]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'mask_labels' : [x['mask_labels']  for x in batch],
        'class_labels': [x['class_labels'] for x in batch],
    }

In [ ]:
BATCH_SIZE = 2

train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 2,
    collate_fn  = collate_fn,
    pin_memory  = True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = 2,
    collate_fn  = collate_fn,
    pin_memory  = True
)

In [ ]:
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    'facebook/mask2former-swin-small-ade-semantic',
    num_labels = NUM_CLASSES,
    id2label   = ID2LABEL,
    label2id   = LABEL2ID,
    ignore_mismatched_sizes = True
)

model = model.to(device)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f' Mask2Former Swin-Small loaded')
print(f'   Total parameters    : {total:,}')
print(f'   Trainable parameters: {trainable:,}')
print(f'   Running on          : {next(model.parameters()).device}')

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 10

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
use_amp = (device.type == 'cuda')
scaler = GradScaler(enabled=use_amp)

metric = evaluate.load('mean_iou')

print(f' Training setup ready')
print(f'   Optimizer  : AdamW (lr=1e-4)')
print(f'   Scheduler  : CosineAnnealingLR')
print(f'   Mixed prec.: {use_amp}')
print(f'   Epochs     : {EPOCHS}')
print(f'   Loss       : computed internally by Mask2Former (classification + mask + dice)')

In [ ]:
history = {'train_loss': [], 'val_miou': []}
best_miou = 0.0
best_model_path = '/content/best_mask2former.pt'

print('Starting training with per-class monitoring...\n')
print(f'{"Epoch":>6}  {"Train Loss":>12}  {"Val mIoU":>10}  {"LR":>10}')
print('-' * 46)

for epoch in range(1, EPOCHS + 1):
  model.train()
  epoch_loss = 0.0

  for batch in train_loader:
    pixel_values = batch['pixel_values'].to(device)
    mask_labels  = [m.to(device) for m in batch['mask_labels']]
    class_labels = [torch.clamp(c.to(device), 0, NUM_CLASSES - 1) for c in batch['class_labels']]

    optimizer.zero_grad()
    with torch.amp.autocast('cuda', enabled=use_amp):
      outputs = model(pixel_values=pixel_values, mask_labels=mask_labels, class_labels=class_labels)
      loss = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    epoch_loss += loss.item()

  avg_loss = epoch_loss / len(train_loader)
  history['train_loss'].append(avg_loss)

  model.eval()
  with torch.no_grad():
    for batch in valid_loader:
      pixel_values = batch['pixel_values'].to(device)
      outputs = model(pixel_values=pixel_values)
      target_sizes = [pixel_values.shape[-2:]] * pixel_values.shape[0]
      preds = processor.post_process_semantic_segmentation(outputs, target_sizes)

      for i, pred in enumerate(preds):
        H, W = pred.shape
        gt_mask = torch.zeros(H, W, dtype=torch.long)
        for mask_t, class_t in zip(batch['mask_labels'][i], batch['class_labels'][i]):
          m = torch.nn.functional.interpolate(mask_t.unsqueeze(0).unsqueeze(0).float(), size=(H, W), mode='nearest').squeeze().long()
          gt_mask[m == 1] = torch.clamp(class_t, 0, NUM_CLASSES - 1).item()

        metric.add_batch(predictions=pred.cpu().numpy()[np.newaxis], references=gt_mask.cpu().numpy()[np.newaxis])

  results = metric.compute(num_labels=NUM_CLASSES, ignore_index=255)
  val_miou = results['mean_iou']
  history['val_miou'].append(val_miou)

  if val_miou > best_miou:
    best_miou = val_miou
    torch.save(model.state_dict(), best_model_path)
    saved = ' (saved)'
  else:
    saved = ''

  lr = scheduler.get_last_lr()[0]
  print(f'{epoch:>6}  {avg_loss:>12.4f}  {val_miou:>10.4f}  {lr:>10.6f}{saved}')

  if epoch % 2 == 0:
      print("    Top per-class IoU:", end=" ")
      sorted_ious = sorted([(ID2LABEL[i], v) for i, v in enumerate(results['per_category_iou']) if not np.isnan(v)], key=lambda x: x[1], reverse=True)
      print(", ".join([f"{n}: {v:.2f}" for n, v in sorted_ious[:5]]))

  scheduler.step()

print(f'\nTraining complete! Best mIoU: {best_miou:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=5)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_miou'], 'm-o', linewidth=2, markersize=5)
best_ep = history['val_miou'].index(max(history['val_miou'])) + 1
axes[1].axvline(x=best_ep, color='red', linestyle='--', alpha=0.7, label=f'Best (epoch {best_ep})')
axes[1].set_title('Validation mIoU', fontsize=14)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Mask2Former Training Results', fontsize=16)
plt.tight_layout(); plt.show()

print(f'Final training loss : {history["train_loss"][-1]:.4f}')
print(f'Best validation mIoU: {best_miou:.4f} at epoch {best_ep}')

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval();

In [ ]:
def predict(image_path, model, processor, device):
  image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
  orig_h, orig_w = image.shape[:2]

  encoded      = processor(images=image, return_tensors='pt')
  pixel_values = encoded['pixel_values'].to(device)

  with torch.no_grad():
    outputs = model(pixel_values=pixel_values)

  pred_mask = processor.post_process_semantic_segmentation(
      outputs,
      target_sizes = [(orig_h, orig_w)]
    )[0].cpu().numpy()

  return image, pred_mask

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

cmap = plt.get_cmap('tab20')
CLASS_COLORS = {i: tuple((np.array(cmap(i % 20)[:3]) * 255).astype(int)) for i in range(NUM_CLASSES)}

def mask_to_color(mask):
    h, w = mask.shape
    color_img = np.zeros((h, w, 3), dtype=np.uint8)
    unique_in_mask = np.unique(mask)
    for cid in unique_in_mask:
        if cid in CLASS_COLORS:
            color_img[mask == cid] = CLASS_COLORS[cid]
    return color_img

In [ ]:
def visualize_predictions(img_dir, mask_dir, model, processor, device, n=4):
    all_files = sorted(os.listdir(img_dir))
    fnames = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg')) and '_mask' not in f][:n]

    fig, axes = plt.subplots(n, 4, figsize=(18, 4 * n))
    if n == 1: axes = [axes]

    for i, fname in enumerate(fnames):
        img_path  = os.path.join(img_dir, fname)
        mask_fname = os.path.splitext(fname)[0] + '_mask.png'
        mask_path = os.path.join(mask_dir, mask_fname)

        if not os.path.exists(mask_path):
            mask_path = os.path.join(mask_dir, fname)

        image, pred  = predict(img_path, model, processor, device)
        gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if gt is None: continue
        gt = np.clip(gt, 0, NUM_CLASSES - 1)

        pred_colored = mask_to_color(pred)
        gt_colored   = mask_to_color(gt)

        overlay = (image.astype(np.float32) * 0.5 + pred_colored.astype(np.float32) * 0.5).clip(0, 255).astype(np.uint8)

        axes[i][0].imshow(image);               axes[i][0].axis('off')
        axes[i][1].imshow(gt_colored);           axes[i][1].axis('off')
        axes[i][2].imshow(pred_colored);         axes[i][2].axis('off')
        axes[i][3].imshow(overlay);             axes[i][3].axis('off')

    titles = ['Original Image', 'Ground Truth', 'Prediction', 'Overlay']
    for j, t in enumerate(titles):
        axes[0][j].set_title(t, fontsize=12, fontweight='bold')

    visible_classes = set()
    for fname in fnames:
        path = os.path.join(mask_dir, os.path.splitext(fname)[0] + '_mask.png')
        if os.path.exists(path):
            m = cv2.imread(path, 0)
            if m is not None: visible_classes.update(np.unique(m))

    patches = [npatches.Patch(color=np.array(CLASS_COLORS[cid])/255, label=ID2LABEL.get(cid, str(cid)))
               for cid in sorted(list(visible_classes)) if cid < NUM_CLASSES]

    fig.legend(handles=patches, loc='lower center', ncol=min(len(patches), 5),
               fontsize=10, title='Visible Classes', bbox_to_anchor=(0.5, -0.08))

    plt.suptitle('Mask2Former Model Predictions', fontsize=16, y=1.02)
    plt.tight_layout(); plt.show()

In [ ]:
visualize_predictions(VALID_IMG_DIR, VALID_MASK_DIR, model, processor, device, n=4)

In [ ]:
model.eval()
metric_final = evaluate.load('mean_iou')

with torch.no_grad():
    for batch in valid_loader:
        pixel_values = batch['pixel_values'].to(device)
        outputs      = model(pixel_values=pixel_values)
        target_sizes = [pixel_values.shape[-2:]] * pixel_values.shape[0]
        preds        = processor.post_process_semantic_segmentation(
            outputs, target_sizes=target_sizes
        )
        for i, pred in enumerate(preds):
            H, W = pred.shape
            gt_mask = torch.zeros(H, W, dtype=torch.long)
            for mask_t, class_t in zip(batch['mask_labels'][i], batch['class_labels'][i]):
                m = torch.nn.functional.interpolate(
                    mask_t.unsqueeze(0).unsqueeze(0).float(),
                    size=(H, W), mode='nearest'
                ).squeeze().long()
                gt_mask[m == 1] = class_t.item()
            metric_final.add_batch(
                predictions = pred.cpu().numpy()[np.newaxis],
                references  = gt_mask.numpy()[np.newaxis]
            )

res = metric_final.compute(num_labels=NUM_CLASSES, ignore_index=255)

print('=' * 42)
print('  Final Evaluation Results')
print('=' * 42)
print(f'  Mean IoU      : {res["mean_iou"]:.4f}')
print(f'  Mean Accuracy : {res["mean_accuracy"]:.4f}')
print()
print('  Per-Class IoU:')
for cid, iou_val in enumerate(res['per_category_iou']):
    name = ID2LABEL.get(cid, f'class_{cid}')
    bar  = '█' * int(iou_val * 20)
    print(f'    {name:>15}: {iou_val:.4f}  {bar}')
print('=' * 42)

In [ ]:
import pandas as pd

def analyze_class_distribution(loader):
    class_counts = {i: 0 for i in range(NUM_CLASSES)}
    total_pixels = 0

    print('Analyzing class distribution in dataset...')
    for batch in loader:
        for labels in batch['class_labels']:
            for cls_id in labels:
                class_counts[cls_id.item()] += 1

    df_dist = pd.DataFrame([
        {'class_id': k, 'label': ID2LABEL[k], 'occurrence_count': v}
        for k, v in class_counts.items()
    ]).sort_values('occurrence_count', ascending=False)

    display(df_dist.head(15))
    return df_dist

train_dist = analyze_class_distribution(train_loader)

In [ ]:
SAVE_DIR = '/content/mask2former_finetuned'
model.load_state_dict(torch.load(best_model_path, map_location=device))

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print(f' Model saved to: {SAVE_DIR}')
print('Files:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f'  {f:40s}  {size:.1f} MB')

import shutil
zip_path = shutil.make_archive('/content/mask2former_finetuned', 'zip', SAVE_DIR)
print(f'\n Download zip: {zip_path}')

In [ ]:
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
import torch, cv2, numpy as np

LOAD_DIR = '/content/mask2former_finetuned'
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loaded_processor = AutoImageProcessor.from_pretrained(LOAD_DIR)
loaded_model     = Mask2FormerForUniversalSegmentation.from_pretrained(LOAD_DIR)
loaded_model     = loaded_model.to(device)
loaded_model.eval()

print(' Model reloaded successfully')
print(f'   Classes: {loaded_model.config.id2label}')

In [ ]:
test_img  = sorted(os.listdir(VALID_IMG_DIR))[0]
img, pred = predict(os.path.join(VALID_IMG_DIR, test_img), loaded_model, loaded_processor, device)
print(f'   Test image shape    : {img.shape}')
print(f'   Predicted mask shape: {pred.shape}')
print(f'   Unique classes found: {np.unique(pred).tolist()}')
print(' Inference working correctly!')